# 03 · Bases de treino e teste + avaliação do FinBERT-PT-BR

Abordagem híbrida:

| base | de onde vem | rótulo |
|---|---|---|
| **teste** (200) | nossas manchetes | sugerido pelo Claude e **revisado por nós** (`teste_revisao.xlsx`) |
| **treino: público** (~500) | [`lucas-leme/Sentiments-FinBERT-PT-BR`](https://huggingface.co/datasets/lucas-leme/Sentiments-FinBERT-PT-BR) | humano (3 anotadores) |
| **treino: FinBERT** (~1.500) | nossas manchetes fora do teste | automático, do FinBERT-PT-BR |

O teste é a única base revisada por humanos sobre o nosso domínio, então **todo modelo do projeto é avaliado nele**. Nenhuma manchete do teste (nem quase-duplicata dela) entra no treino.

O FinBERT-PT-BR é avaliado aqui como **linha de base**. Ele foi treinado com o dataset público, então não faz sentido avaliá-lo nessa base: por isso a avaliação usa só o nosso teste.

**Como rodar no Colab:** GPU T4, depois *Executar tudo*. Antes, suba a planilha revisada `teste_revisao.xlsx` para `MyDrive/portfolio-radar/`.

In [ ]:
try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

from pathlib import Path

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = Path("/content/drive/MyDrive/portfolio-radar")
else:
    PASTA = Path("raw")

In [ ]:
import re
import pandas as pd

ROTULOS = ["boa", "neutra", "ruim"]
URL_PUBLICO = "https://huggingface.co/datasets/lucas-leme/Sentiments-FinBERT-PT-BR/resolve/main/sentiments.csv"
SIMILARIDADE_MAX = 0.6  # acima disso, a manchete do treino é tratada como duplicata de uma do teste

## Base de teste (revisada)

In [ ]:
teste = pd.read_excel(PASTA / "teste_revisao.xlsx")
teste["rotulo"] = teste["rotulo"].str.strip().str.lower()

invalidos = teste[~teste["rotulo"].isin(ROTULOS + ["descartar"])]
assert invalidos.empty, f"Rótulos inválidos nas linhas: {invalidos['id'].tolist()}"

teste = teste[teste["rotulo"] != "descartar"].reset_index(drop=True)
print(f"\nTeste final: {len(teste)} manchetes")
teste["rotulo"].value_counts().to_frame("manchetes")

## Treino: dataset público

In [ ]:
MAPA_PUBLICO = {"Positivo": "boa", "Negativo": "ruim", "Neutro": "neutra"}  # "Não se aplica" sai

publico = pd.read_csv(URL_PUBLICO, index_col=0)
publico = publico[publico["sentiment"].isin(MAPA_PUBLICO)]
publico = pd.DataFrame({
    "texto": publico["text"],
    "rotulo": publico["sentiment"].map(MAPA_PUBLICO),
    "fonte": "publico",
})
print(f"{len(publico)} textos")
publico["rotulo"].value_counts().to_frame("textos")

## Treino: nossas manchetes com rótulo do FinBERT

Parte do `manchetes_filtradas.csv` (notebook 01) e tira:
- as manchetes do teste (pelo link);
- quase-duplicatas do teste: a mesma notícia com outra redação ("Petrobras atinge R$ 800 bi…" aparece várias vezes). Medimos a sobreposição de palavras (Jaccard) e cortamos acima de `SIMILARIDADE_MAX`.

In [ ]:
def palavras(texto):
    return set(re.findall(r"\w+", texto.lower()))


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0


manchetes = pd.read_csv(PASTA / "manchetes_filtradas.csv")
manchetes = manchetes[~manchetes["link"].isin(teste["link"])]

palavras_teste = [palavras(t) for t in teste["manchete"]]
sim_max = manchetes["manchete"].apply(lambda m: max(jaccard(palavras(m), p) for p in palavras_teste))
quase_dup = sim_max > SIMILARIDADE_MAX
print(f"Removidas {quase_dup.sum()} quase-duplicatas do teste. Exemplos:")
for m in manchetes.loc[quase_dup, "manchete"].head(5):
    print(" -", m)

manchetes = manchetes[~quase_dup].reset_index(drop=True)
print(f"\nSobram {len(manchetes)} manchetes para rotular com o FinBERT")

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
print("GPU" if device == 0 else "CPU (mais lento, mas funciona)")
clf = pipeline("text-classification", model="lucas-leme/FinBERT-PT-BR", device=device)
MAPA_FINBERT = {"POSITIVE": "boa", "NEGATIVE": "ruim", "NEUTRAL": "neutra"}


def classifica(textos):
    saidas = clf(list(textos), batch_size=32, truncation=True)
    return [MAPA_FINBERT[s["label"]] for s in saidas], [round(s["score"], 3) for s in saidas]


rot, conf = classifica(manchetes["manchete"])
finbert = pd.DataFrame({"texto": manchetes["manchete"], "rotulo": rot, "fonte": "finbert", "confianca": conf})
finbert["rotulo"].value_counts().to_frame("manchetes")

## Salva as bases

Os próximos notebooks (rede neural, Transformer) treinam em `treino.csv` e avaliam em `teste.csv`.

In [ ]:
treino = pd.concat([publico, finbert], ignore_index=True)
treino.to_csv(PASTA / "treino.csv", index=False)
teste_final = teste[["id", "empresa", "manchete", "rotulo"]].rename(columns={"manchete": "texto"})
teste_final.to_csv(PASTA / "teste.csv", index=False)

print(f"treino.csv: {len(treino)} textos")
print(f"teste.csv:  {len(teste_final)} manchetes")
pd.crosstab(treino["fonte"], treino["rotulo"], margins=True)

## Avaliação do FinBERT-PT-BR no teste (linha de base)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

teste["previsto"], teste["confianca"] = classifica(teste["manchete"])

print(f"Acurácia: {accuracy_score(teste['rotulo'], teste['previsto']):.3f}\n")
print(classification_report(teste["rotulo"], teste["previsto"], labels=ROTULOS, digits=3, zero_division=0))

In [ ]:
pd.DataFrame(
    confusion_matrix(teste["rotulo"], teste["previsto"], labels=ROTULOS),
    index=[f"real: {r}" for r in ROTULOS],
    columns=[f"previsto: {r}" for r in ROTULOS],
)

## Exemplos por classe

Para cada rótulo real, alguns acertos e alguns erros do FinBERT no nosso teste.

In [ ]:
pd.set_option("display.max_colwidth", 120)
for r in ROTULOS:
    da_classe = teste[teste["rotulo"] == r]
    acertos = da_classe[da_classe["previsto"] == r].sample(min(3, (da_classe["previsto"] == r).sum()), random_state=0)
    erros_r = da_classe[da_classe["previsto"] != r].sample(min(2, (da_classe["previsto"] != r).sum()), random_state=0)
    print(f"=== real: {r} ===")
    for _, l in pd.concat([acertos, erros_r]).iterrows():
        marca = "OK  " if l["previsto"] == r else "ERRO"
        print(f"{marca} previsto={l['previsto']:<6} ({l['confianca']:.2f})  {l['manchete']}")
    print()

### Teste com manchetes escritas por nós

Manchetes inventadas, uma ou mais por classe, para ver o comportamento do modelo em casos controlados. Inclui o exemplo da proposta, em que o tom da frase e o efeito para o acionista apontam para lados diferentes.

In [ ]:
exemplos = pd.DataFrame([
    ("Itaú lucra R$ 12 bilhões no trimestre e supera expectativas", "boa"),
    ("Petrobras anuncia pagamento de R$ 20 bilhões em dividendos", "boa"),
    ("Petrobras reajusta o preço da gasolina nas refinarias", "boa"),        # ruim para o consumidor, boa para o acionista
    ("Bradesco é condenado a pagar multa milionária ao Procon", "ruim"),
    ("Ações da Ambev despencam após resultado abaixo do esperado", "ruim"),
    ("Localiza corta 2 mil vagas e fecha lojas para reduzir custos", "boa"),  # notícia "triste", mas corte de custos tende a agradar o acionista
    ("WEG abre inscrições para programa de estágio", "neutra"),
    ("Vale realiza assembleia de acionistas nesta sexta-feira", "neutra"),
], columns=["manchete", "esperado"])

exemplos["previsto"], exemplos["confianca"] = classifica(exemplos["manchete"])
exemplos["acertou"] = exemplos["esperado"] == exemplos["previsto"]
exemplos

## Análise dos erros

In [ ]:
erros = teste[teste["rotulo"] != teste["previsto"]]
print(f"{len(erros)} erros de {len(teste)}")
pd.set_option("display.max_colwidth", 120)
erros[["manchete", "rotulo", "previsto", "confianca"]].sort_values("confianca", ascending=False)

In [ ]:
# Acerto por empresa: mostra onde o modelo tem mais dificuldade
teste.assign(acertou=teste["rotulo"] == teste["previsto"]).groupby("empresa")["acertou"].mean().sort_values().to_frame("acurácia")

## Salva o resultado em Excel

`resultado_finbert.xlsx` vai para a mesma pasta do Drive, com uma aba por tabela, para levar os números ao relatório.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f, n = precision_recall_fscore_support(teste["rotulo"], teste["previsto"], labels=ROTULOS, zero_division=0)
metricas = pd.DataFrame({"precisao": p, "revocacao": r, "f1": f, "suporte": n}, index=ROTULOS).round(3)
metricas.loc["macro"] = [round(p.mean(), 3), round(r.mean(), 3), round(f.mean(), 3), n.sum()]
metricas.loc["acuracia"] = [None, None, round(accuracy_score(teste["rotulo"], teste["previsto"]), 3), n.sum()]

matriz = pd.DataFrame(
    confusion_matrix(teste["rotulo"], teste["previsto"], labels=ROTULOS),
    index=[f"real: {r}" for r in ROTULOS], columns=[f"previsto: {r}" for r in ROTULOS],
)
por_empresa = (teste.assign(acertou=teste["rotulo"] == teste["previsto"])
               .groupby("empresa")["acertou"].agg(["mean", "size"])
               .rename(columns={"mean": "acuracia", "size": "manchetes"}).round(3).sort_values("acuracia"))
previsoes = teste[["id", "empresa", "manchete", "rotulo", "previsto", "confianca"]].assign(
    acertou=teste["rotulo"] == teste["previsto"])

arq = PASTA / "resultado_finbert.xlsx"
with pd.ExcelWriter(arq) as xl:
    metricas.to_excel(xl, sheet_name="metricas")
    matriz.to_excel(xl, sheet_name="matriz_confusao")
    por_empresa.to_excel(xl, sheet_name="por_empresa")
    previsoes.to_excel(xl, sheet_name="previsoes", index=False)
    exemplos.to_excel(xl, sheet_name="exemplos_nossos", index=False)
print(f"Salvo em {arq}")
metricas

## Análise dos resultados

> **Preencher depois de rodar com o teste revisado.** Os números abaixo entre colchetes vêm das células acima. Os pontos em itálico foram observados num teste preliminar e precisam ser confirmados.

**Resumo.** O FinBERT-PT-BR, usado sem ajuste, acertou [acurácia] das [N] manchetes do teste, com F1 macro de [F1 macro]. A classe com melhor desempenho foi [classe] e a pior foi [classe].

**Onde o modelo erra.**
- *Tende a classificar como `ruim` notícias que consideramos neutras: obras, manutenção, decisões judiciais ainda em andamento.* [confirmar e citar 1–2 exemplos]
- *Perde parte das notícias `boa`, rotulando-as como neutras.* [confirmar pela revocação de `boa`]
- Empresas com mais erros: [empresas da tabela de acerto por empresa].

**Por que.** O FinBERT foi treinado para avaliar o tom do texto financeiro, enquanto o nosso critério é o impacto provável para o acionista. Os dois divergem quando a frase tem tom negativo e efeito positivo para a empresa (reajuste de preço, corte de custos), ou quando a notícia é operacional e não afeta o resultado. [citar o que aconteceu com as manchetes escritas por nós]

**Limitações.**
- O teste tem só [N] manchetes, com poucas `ruim` ([n ruim]); as métricas dessa classe variam muito com poucos erros.
- Os rótulos do teste foram sugeridos por IA generativa e revisados por nós, o que pode puxar a revisão para concordar com a sugestão.
- O modelo só vê a manchete, sem o texto da matéria nem o tamanho da empresa.

**Próximos passos.** Os modelos das próximas etapas (rede neural e Transformer com fine-tuning) serão treinados em `treino.csv` e avaliados neste mesmo teste, com o FinBERT como linha de base.